In [7]:
import os
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer, PageBreak

OUT = 'D2_daily_watch_time'
ORDER = ['Less than 1 hour', '1 hour', '2 hours', '3 hours', 'More than that']
RAW_SKIP = {'A1_age', 'B2_family_time_hours', 'C1_height', 'C2_weight', 'D1_years_watching'}
BASE = r'E:\Important\Skills\Reels-Mental-Health-Well-Being-Report'
PDF_PATH = os.path.join(BASE, 'cross_tabulation', 'cross_tabulation_D2_daily_watch_time.pdf')

d = pd.read_csv(os.path.join(BASE, 'dataset', 'dataset_v1.csv'))
d[OUT] = pd.Categorical(d[OUT], categories=ORDER, ordered=True)
preds = [c for c in d.columns if c != OUT and c not in RAW_SKIP]

# Cross-tabulations + chi-square tests
results = {}
for c in preds:
    ct = pd.crosstab(d[c], d[OUT])
    chi, p, dof, _ = chi2_contingency(ct)
    results[c] = dict(ct=ct, pct=ct.div(ct.sum(1), axis=0) * 100, chi=chi, dof=dof, p=p)

summary_df = pd.DataFrame(
    [(c, r['chi'], r['dof'], r['p'], r['p'] < 0.05) for c, r in results.items()],
    columns=['Variable', 'Chi-square', 'df', 'p-value', 'Significant'])
summary_df

,Variable,Chi-square,df,p-value,Significant
0,A2_gender,7.348792,4,1.185648e-01,False
1,A3_religion,3.487162,4,4.798331e-01,False
2,A4_marital_status,16.985381,8,3.026167e-02,True
3,A5_residence,3.427874,4,4.889296e-01,False
4,A6_occupation,15.463670,12,2.170533e-01,False
5,B1_family_type,15.746386,12,2.031301e-01,False
6,B3_family_relationship,26.978072,16,4.172752e-02,True
7,B4_in_person_feeling,34.123972,16,5.227857e-03,True
8,C3_sitting_duration,267.062564,12,3.743516e-50,True
9,C4_physical_problems,45.430159,12,8.694902e-06,True


In [8]:
# Preview one cross-tab
c = preds[0]
display(results[c]['ct'], results[c]['pct'].round(1))

D2_daily_watch_time,Less than 1 hour,1 hour,2 hours,3 hours,More than that
A2_gender,,,,,
Female,42,31,36,22,38
Male,68,77,99,36,57


D2_daily_watch_time,Less than 1 hour,1 hour,2 hours,3 hours,More than that
A2_gender,,,,,
Female,24.9,18.3,21.3,13.0,22.5
Male,20.2,22.8,29.4,10.7,16.9


In [9]:
ss = getSampleStyleSheet()

def tstyle(header_color):
    return TableStyle([('BACKGROUND', (0, 0), (-1, 0), header_color), ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 8), ('GRID', (0, 0), (-1, -1), 0.4, colors.grey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'), ('ALIGN', (1, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f2f5f9')])])

el = [Paragraph('Cross-Tabulation Report', ss['Title']),
      Paragraph(f'Outcome variable: <b>{OUT}</b> (columns) vs. each variable (rows). N = {len(d)}. '
                'Cells show count (row %). Chi-square test of independence per variable.', ss['Normal']),
      Spacer(1, 10), Paragraph('Summary of association tests (* p &lt; 0.05)', ss['Heading2'])]

srows = [['Variable', 'Chi-square', 'df', 'p-value', 'Sig.']] + [
    [row['Variable'], f"{row['Chi-square']:.2f}", str(row['df']), f"{row['p-value']:.4f}", '*' if row['Significant'] else '']
    for _, row in summary_df.iterrows()]
t = Table(srows, repeatRows=1, colWidths=[260, 80, 50, 80, 40]); t.setStyle(tstyle(colors.HexColor('#2c3e50')))
el += [t, PageBreak()]

for c, r in results.items():
    ct, pct = r['ct'], r['pct']
    rows = [[c] + ORDER + ['Total']]
    for idx in ct.index:
        rows.append([str(idx)] + [f'{ct.loc[idx, o]} ({pct.loc[idx, o]:.1f}%)' for o in ORDER] + [str(ct.loc[idx].sum())])
    tot = ct.sum()
    rows.append(['Total'] + [f'{tot[o]} ({tot[o]/len(d)*100:.1f}%)' for o in ORDER] + [str(len(d))])
    tb = Table(rows, colWidths=[170] + [100] * 5 + [50], repeatRows=1); tb.setStyle(tstyle(colors.HexColor('#1f6f8b')))
    el += [Paragraph(f'{c} × {OUT}', ss['Heading3']), tb,
           Paragraph(f"Chi-square = {r['chi']:.2f}, df = {r['dof']}, p = {r['p']:.4f}" + (' (significant)' if r['p'] < 0.05 else ''), ss['Normal']),
           Spacer(1, 14)]

os.makedirs('cross_tabulation', exist_ok=True)
SimpleDocTemplate(PDF_PATH, pagesize=landscape(A4), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=30).build(el)
print('Saved', PDF_PATH, '-', len(results), 'cross-tabs')

Saved E:\Important\Skills\Reels-Mental-Health-Well-Being-Report\cross_tabulation\cross_tabulation_D2_daily_watch_time.pdf - 40 cross-tabs
